# 01 — Otopsi: Ekim 2024 Saatlik Toplu Ulaşım Verisi

**Soru:** İBB'nin saatlik toplu ulaşım verisi ne anlatıyor, ne kadar kirli, neye güvenebiliriz?

Kaynak: [İBB Açık Veri — Saatlik Toplu Ulaşım Veri Seti](https://data.ibb.gov.tr/dataset/hourly-public-transport-data-set) (BELBİM A.Ş.)

Bulguların özeti en alttaki **Otopsi Raporu** bölümünde.

In [1]:
import duckdb

# Sabitler — tüm yollar ve parametreler burada
CSV = "../veri/ham/hourly_transportation_202410.csv"
BEKLENEN_GUN = 31  # Ekim

## 1) Genel bakış: satır sayısı, kolonlar, örnek satırlar

In [2]:
duckdb.sql(f"SELECT COUNT(*) AS satir_sayisi FROM '{CSV}'").show()
duckdb.sql(f"DESCRIBE SELECT * FROM '{CSV}'").show()
duckdb.sql(f"SELECT * FROM '{CSV}' LIMIT 5").show(max_width=200)

┌──────────────┐
│ satir_sayisi │
│    int64     │
├──────────────┤
│      2770719 │
└──────────────┘

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ transition_date       │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ transition_hour       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ transport_type_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ road_type             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ line                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ transfer_type         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ number_of_passage     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ number_of_passe

┌─────────────────┬─────────────────┬───────────────────┬───────────┬──────────────────────────────────────────┬───┬──────────────┬──────────────────────┬───────────┬───────────┬─────────────────────┐
│ transition_date │ transition_hour │ transport_type_id │ road_type │                   line                   │ … │ product_kind │ transaction_type_de… │   town    │ line_name │ station_poi_desc_cd │
│      date       │     varchar     │       int64       │  varchar  │                 varchar                  │ … │   varchar    │       varchar        │  varchar  │  varchar  │       varchar       │
├─────────────────┼─────────────────┼───────────────────┼───────────┼──────────────────────────────────────────┼───┼──────────────┼──────────────────────┼───────────┼───────────┼─────────────────────┤
│ 2024-10-01      │ 00              │                 1 │ OTOYOL    │ CEBECI - TAKSIM                          │ … │ TAM          │ Tam Kontur           │ SARIYER   │ 36T       │ NULL             

## 2) Zaman kapsamı: ayın tamamı var mı?

Ekim'in 31 günü × 24 saat = 744 gün-saat kombinasyonu bekliyoruz.

In [3]:
duckdb.sql(f"""
SELECT MIN(transition_date) AS ilk_gun, MAX(transition_date) AS son_gun,
       COUNT(DISTINCT transition_date) AS gun_sayisi,
       COUNT(DISTINCT (transition_date, transition_hour)) AS dolu_kombinasyon,
       {BEKLENEN_GUN} * 24 AS beklenen_kombinasyon
FROM '{CSV}'
""").show()

┌────────────┬────────────┬────────────┬──────────────────┬──────────────────────┐
│  ilk_gun   │  son_gun   │ gun_sayisi │ dolu_kombinasyon │ beklenen_kombinasyon │
│    date    │    date    │   int64    │      int64       │        int32         │
├────────────┼────────────┼────────────┼──────────────────┼──────────────────────┤
│ 2024-10-01 │ 2024-10-18 │         18 │              138 │                  744 │
└────────────┴────────────┴────────────┴──────────────────┴──────────────────────┘



In [4]:
# Gün bazında saat kapsamı — hangi günler tam?
duckdb.sql(f"""
SELECT transition_date,
       COUNT(DISTINCT transition_hour) AS dolu_saat,
       COUNT(*) AS satir,
       SUM(number_of_passenger) AS yolcu
FROM '{CSV}'
GROUP BY 1 ORDER BY 1
""").show(max_rows=40)

┌─────────────────┬───────────┬────────┬─────────┐
│ transition_date │ dolu_saat │ satir  │  yolcu  │
│      date       │   int64   │ int64  │ int128  │
├─────────────────┼───────────┼────────┼─────────┤
│ 2024-10-01      │         1 │    720 │    1495 │
│ 2024-10-02      │        24 │ 550280 │ 7515377 │
│ 2024-10-03      │         1 │    837 │    1898 │
│ 2024-10-04      │        24 │ 562053 │ 7601373 │
│ 2024-10-05      │         1 │   1208 │    2952 │
│ 2024-10-06      │         3 │   1356 │    3575 │
│ 2024-10-07      │         1 │    843 │    2039 │
│ 2024-10-08      │         4 │    822 │    1774 │
│ 2024-10-09      │         1 │    888 │    1918 │
│ 2024-10-10      │         1 │    771 │    1793 │
│ 2024-10-11      │         1 │    884 │    2059 │
│ 2024-10-12      │         1 │   1279 │    3058 │
│ 2024-10-13      │         1 │    927 │    2146 │
│ 2024-10-14      │         1 │    694 │    1601 │
│ 2024-10-15      │         1 │    765 │    1594 │
│ 2024-10-16      │        24 │

### Bulgu 1 — DOSYA EKSİK (portal kaynaklı)

- Dosyada sadece **1–18 Ekim** var; onun da yalnızca **5 günü tam** (2, 4, 16, 17, 18 Ekim).
- Diğer günlerde neredeyse sadece `00` saati var, o da cılız (~700-1300 satır; tam bir gecenin 00 saati ~binlerce satır olmalı).
- İndirilen dosya portaldaki ile **bayt bayt aynı** (274.570.733 bayt, CKAN API ile doğrulandı) → sorun indirmede değil, **portalın kendi dosyasında**.

**Sonuç:** Her ay dosyası kullanılmadan önce `scriptler/validate_csv.py` ile doğrulanmalı. Analize girecek aylar tam kapsamlı olanlardan seçilmeli.

## 3) Ulaşım türleri: `transport_type_id` ↔ `road_type`

In [5]:
duckdb.sql(f"""
SELECT transport_type_id, road_type,
       COUNT(*) AS satir, SUM(number_of_passenger) AS yolcu
FROM '{CSV}' GROUP BY 1,2 ORDER BY 1
""").show()

┌───────────────────┬───────────┬─────────┬──────────┐
│ transport_type_id │ road_type │  satir  │  yolcu   │
│       int64       │  varchar  │  int64  │  int128  │
├───────────────────┼───────────┼─────────┼──────────┤
│                 1 │ OTOYOL    │ 1830279 │ 20589703 │
│                 2 │ RAYLI     │  848666 │ 16741815 │
│                 3 │ DENİZ     │   91774 │   730375 │
└───────────────────┴───────────┴─────────┴──────────┘



### Bulgu 2 — Üç tür, bire bir eşleşme

| id | road_type | anlamı |
|----|-----------|--------|
| 1 | OTOYOL | karayolu (İETT otobüs, metrobüs vb.) |
| 2 | RAYLI | metro, tramvay, Marmaray |
| 3 | DENİZ | vapur, deniz motoru |

`road_type` içinde Türkçe karakter var (`DENİZ`) — filtrelerken buna dikkat.

## 4) `town` kolonu: ilçe analizi için güvenilir mi?

In [6]:
# İlçe sayısı ve dağılım
duckdb.sql(f"SELECT COUNT(DISTINCT town) AS ilce_sayisi FROM '{CSV}'").show()
duckdb.sql(f"""
SELECT town, COUNT(*) AS satir, SUM(number_of_passenger) AS yolcu
FROM '{CSV}' GROUP BY 1 ORDER BY yolcu DESC LIMIT 10
""").show()

┌─────────────┐
│ ilce_sayisi │
│    int64    │
├─────────────┤
│          39 │
└─────────────┘



┌──────────────┬────────┬──────────┐
│     town     │ satir  │  yolcu   │
│   varchar    │ int64  │  int128  │
├──────────────┼────────┼──────────┤
│ BAKIRKOY     │ 769606 │ 11248092 │
│ FATIH        │ 243188 │  3759894 │
│ USKUDAR      │  78451 │  1996734 │
│ KUCUKCEKMECE │ 186224 │  1852216 │
│ SISLI        │  52390 │  1808979 │
│ KADIKOY      │  77306 │  1627083 │
│ SARIYER      │ 186652 │  1216465 │
│ NULL         │  68815 │  1210794 │
│ KARTAL       │ 151117 │  1163953 │
│ ATASEHIR     │ 154402 │  1012300 │
└──────────────┴────────┴──────────┘
  10 rows                3 columns



In [7]:
# RAYLI'da town istasyonun ilçesi mi? (M2 örneği)
duckdb.sql(f"""
SELECT line_name, station_poi_desc_cd, town, SUM(number_of_passenger) AS yolcu
FROM '{CSV}' WHERE line_name = 'M2'
GROUP BY 1,2,3 ORDER BY yolcu DESC LIMIT 8
""").show()

┌───────────┬─────────────────────┬─────────┬────────┐
│ line_name │ station_poi_desc_cd │  town   │ yolcu  │
│  varchar  │       varchar       │ varchar │ int128 │
├───────────┼─────────────────────┼─────────┼────────┤
│ M2        │ SISLI 2 KUZEY       │ SISLI   │ 305311 │
│ M2        │ YENIKAPI KUZEY      │ FATIH   │ 222850 │
│ M2        │ TAKSIM GUNEY        │ BEYOGLU │ 180364 │
│ M2        │ GAYRETTEPE          │ SISLI   │ 179978 │
│ M2        │ YENIKAPI GUNEY      │ FATIH   │ 166196 │
│ M2        │ OSMANBEY 2 GUNEY    │ SISLI   │ 121879 │
│ M2        │ OSMANBEY KUZEY      │ SISLI   │ 119544 │
│ M2        │ LEVENT GUNEY        │ SISLI   │ 112183 │
└───────────┴─────────────────────┴─────────┴────────┘



In [8]:
# OTOYOL'da bir hat kaç farklı town'a yazılıyor? (25G örneği: Sarıyer-Taksim hattı)
duckdb.sql(f"""
SELECT line_name, line, town, COUNT(*) AS satir
FROM '{CSV}' WHERE line_name = '25G' GROUP BY 1,2,3
""").show(max_width=200)

# town'u NULL olan satırlar hangi hatlarda?
duckdb.sql(f"""
SELECT road_type, line_name, COUNT(*) AS satir, SUM(number_of_passenger) AS yolcu
FROM '{CSV}' WHERE town IS NULL OR TRIM(town) = ''
GROUP BY 1,2 ORDER BY yolcu DESC LIMIT 5
""").show()

┌───────────┬──────────────────────────────────────┬───────────┬───────┐
│ line_name │                 line                 │   town    │ satir │
│  varchar  │               varchar                │  varchar  │ int64 │
├───────────┼──────────────────────────────────────┼───────────┼───────┤
│ 25G       │ SARIYER-HACIOSMAN-MECIDIYEKOY-TAKSIM │ KAGITHANE │   438 │
└───────────┴──────────────────────────────────────┴───────────┴───────┘



┌───────────┬───────────┬───────┬─────────┐
│ road_type │ line_name │ satir │  yolcu  │
│  varchar  │  varchar  │ int64 │ int128  │
├───────────┼───────────┼───────┼─────────┤
│ RAYLI     │ M7        │ 68815 │ 1210794 │
└───────────┴───────────┴───────┴─────────┘



### Bulgu 3 — `town` iki farklı anlam taşıyor

- 39 farklı değer = İstanbul'un 39 ilçesi. Kulağa mükemmel geliyor ama:
- **RAYLI:** `town` = istasyonun gerçek ilçesi (M2'de Şişli/Fatih/Beyoğlu doğru eşleşiyor). ✅ Coğrafi analizde kullanılabilir.
- **OTOYOL:** her hat tek bir `town`a sabitlenmiş (25G Sarıyer-Taksim hattı → hep KAGITHANE). Bu büyük ihtimalle **garaj/işletme bölgesi**, yolculuğun yapıldığı yer DEĞİL. BAKIRKOY'un anormal payı (~10M yolcu) bundan. ❌ Otobüste ilçe analizi için KULLANILMAZ.
- **M7 metrosunun tamamında `town` NULL** (~1.2M yolcu) — istasyon→ilçe eşlemesi portal tarafında eksik. İlçe analizinde M7 için elle eşleme gerekecek (`veri/esleme/`).

## 5) `number_of_passage` vs `number_of_passenger`

In [9]:
duckdb.sql(f"""
SELECT SUM(number_of_passage) AS gecis,
       SUM(number_of_passenger) AS yolcu,
       COUNT(*) FILTER (number_of_passage != number_of_passenger) AS farkli_satir,
       COUNT(*) AS toplam_satir
FROM '{CSV}'
""").show()

┌──────────┬──────────┬──────────────┬──────────────┐
│  gecis   │  yolcu   │ farkli_satir │ toplam_satir │
│  int128  │  int128  │    int64     │    int64     │
├──────────┼──────────┼──────────────┼──────────────┤
│ 39060183 │ 38061893 │       215417 │      2770719 │
└──────────┴──────────┴──────────────┴──────────────┘



### Bulgu 4 — Geçiş ≥ yolcu

- ~39,1M geçiş vs ~38,1M yolcu; satırların ~%8'inde ikisi farklı.
- Yorum: `number_of_passage` = turnike/validatör geçişi (aynı kişi grup ödemesi vb. ile birden çok geçebilir), `number_of_passenger` = kişi sayısı.
- **Karar: analizlerde `number_of_passenger` kullanılacak** ("kaç kişi yolculuk etti" sorusuna daha yakın).

## 6) NULL / boşluk haritası

In [10]:
duckdb.sql(f"""
SELECT road_type, COUNT(*) AS satir,
       COUNT(*) FILTER (station_poi_desc_cd IS NULL OR TRIM(station_poi_desc_cd) = '') AS istasyon_bos,
       COUNT(*) FILTER (town IS NULL OR TRIM(town) = '') AS town_bos,
       COUNT(*) FILTER (line_name IS NULL OR TRIM(line_name) = '') AS hat_bos
FROM '{CSV}' GROUP BY 1
""").show()

┌───────────┬─────────┬──────────────┬──────────┬─────────┐
│ road_type │  satir  │ istasyon_bos │ town_bos │ hat_bos │
│  varchar  │  int64  │    int64     │  int64   │  int64  │
├───────────┼─────────┼──────────────┼──────────┼─────────┤
│ DENİZ     │   91774 │         1919 │        0 │       0 │
│ OTOYOL    │ 1830279 │      1691456 │        0 │       0 │
│ RAYLI     │  848666 │         2867 │    68815 │       0 │
└───────────┴─────────┴──────────────┴──────────┴─────────┘



### Bulgu 5 — Granülarite türe göre değişiyor

- **RAYLI:** istasyon bazında (`station_poi_desc_cd` dolu) → en zengin coğrafi çözünürlük.
- **OTOYOL:** çoğunlukla hat bazında (istasyon ~%92 boş) → coğrafya için hat güzergâhı gerekir.
- **DENİZ:** iskele bazında büyük oranda dolu.

## Otopsi Raporu (özet)

1. **Portalın Ekim 2024 dosyası eksik** — sadece 5 tam gün (2, 4, 16, 17, 18 Ekim). Her ay indirilince `scriptler/validate_csv.py` ile kontrol şart.
2. Şema temiz: 13 kolon, İngilizce adlar, tarih ISO formatında; metinler transliterasyonlu BÜYÜK HARF (tek istisna: `DENİZ`).
3. `town` raylı sistemde güvenilir, otobüste garaj bölgesi — otobüs ilçe analizi için kullanılamaz. M7'de tamamen NULL.
4. Metrik kararı: `number_of_passenger`.
5. Granülarite: raylı=istasyon, otobüs=hat, deniz=iskele.

**Sonraki adım:** Diğer ayları indirip validate etmek; tam kapsamlı aylardan analiz dönemini seçmek (Faz 2'ye giriş).